In [1]:
# 1. Install Python Packages (LlamaIndex, ChromaDB, Streamlit)
!pip install -q llama-index-core \
                llama-index-llms-ollama \
                llama-index-embeddings-huggingface \
                llama-index-vector-stores-chroma \
                llama-index-readers-file \
                chromadb \
                streamlit

# 2. Install Linux System Tools (pciutils for GPU detection and zstd for Ollama installation)
!apt-get update && apt-get install -y pciutils zstd > /dev/null

# 3. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n✅ Installation Complete! System is ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3

In [2]:
import subprocess
import time
import os

# 1. Kill any existing ollama to start fresh
!pkill -9 ollama
time.sleep(2)

# 2. Start the server (using DEVNULL to keep the screen clean)
print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Wait 20 seconds (Server load aaga time edukum)
print("Waiting for server to initialize (20s)...")
time.sleep(20)

# 4. Pull the model (Explicitly run and wait for it)
print("Pulling Qwen 2.5 Model...")
!ollama pull qwen2.5:1.5b

# 5. Final check to see if it's actually there
print("\n--- Verification ---")
check = os.popen("ollama list").read()
if "qwen2.5:1.5b" in check:
    print(" SUCCESS: Qwen 2.5 is loaded and ready!")
else:
    print("ERROR: Model not found. Check your internet connection.")

Starting Ollama server...
Waiting for server to initialize (20s)...
Pulling Qwen 2.5 Model...


--- Verification ---
 SUCCESS: Qwen 2.5 is loaded and ready!


In [3]:
!curl http://127.0.0.1:11434

Ollama is running

In [4]:
%%writefile backend_logic.py
import os
import shutil
import chromadb
import uuid
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, StorageContext, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

def init_settings():
    # Models setup
    Settings.llm = Ollama(model="qwen2.5:1.5b", base_url="http://127.0.0.1:11434", request_timeout=600.0)
    Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

def process_documents(data_path="./data"):
    init_settings()

    # 1. ALWAYS start with a fresh folder name to kill old memory
    unique_id = str(uuid.uuid4())[:8]
    db_path = f"./chroma_db_{unique_id}"

    # 2. Setup fresh ChromaDB
    db = chromadb.PersistentClient(path=db_path)
    chroma_collection = db.get_or_create_collection("rag_collection")
    vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

    # 3. Load ONLY current files from the folder
    if not os.path.exists(data_path): os.makedirs(data_path)
    documents = SimpleDirectoryReader(data_path).load_data()

    if not documents:
        raise ValueError("No documents found in folder!")

    # 4. Build fresh index
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True
    )
    return index

Writing backend_logic.py


In [5]:
!pip install streamlit

In [48]:
%%writefile app.py
import streamlit as st
import os
import subprocess
import time
from backend_logic import process_documents, init_settings
from llama_index.core.prompts import PromptTemplate

# 1. Background System Core - Ollama Initialization
@st.cache_resource
def start_ollama_and_model():
    try:
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(4)
        subprocess.run(["ollama", "pull", "qwen2.5:1.5b"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return True
    except:
        return False

# Initialize Page Settings
st.set_page_config(page_title="Qwen AI Assistant", page_icon="🩵", layout="wide")
ollama_status = start_ollama_and_model()

if 'initialized' not in st.session_state:
    init_settings()
    st.session_state.initialized = True

# 2. MATCHING EXACT CODES FOR LOGO STYLE AND PREMIUM SIDEBAR
st.markdown("""
    <style>
    /* Fixed Top Spacing - Removes empty padding at the very top */
    .block-container {
        padding-top: 1.5rem !important;
        padding-bottom: 0rem !important;
    }

    /* 100% Match for Sidebar Warning Box from Image */
    .image-style-warning-box {
        background-color: #111a24;
        padding: 15px;
        border-radius: 8px;
        border: 1px solid #1c2d3d;
        margin-top: 15px;
        margin-bottom: 15px;
    }
    .image-style-warning-title {
        color: #ffb700; /* Yellow Lightning Bolt Color */
        font-size: 0.9rem;
        margin: 0;
        font-weight: 600;
        display: flex;
        align-items: center;
        gap: 6px;
    }
    .image-style-warning-text {
        color: #8b949e;
        font-size: 0.82rem;
        margin: 8px 0 0 0;
        line-height: 1.4;
    }

    /* PREMIUM SIDEBAR EXTRA STYLING CODES */
    /* FIXED: Changed Context Ingestion Color to Premium Matte Grey */
    .sidebar-header-premium {
        color: #e2e8f0;
        font-size: 1.25rem;
        font-weight: 600;
        margin-bottom: 2px;
        font-family: 'Inter', sans-serif;
    }
    .sidebar-subheader-premium {
        color: #8b949e;
        font-size: 0.85rem;
        margin-top: 0px;
        margin-bottom: 15px;
    }

    /* Custom Live Status Badge styling */
    .status-badge-container {
        background-color: #161b22;
        padding: 12px;
        border-radius: 8px;
        border: 1px solid #21262d;
        margin-top: 10px;
        margin-bottom: 15px;
    }

    /* Styling Streamlit native file uploader widget container */
    div[data-testid="stFileUploader"] {
        background-color: #0d1117 !important;
        border: 1px dashed #30363d !important;
        border-radius: 8px !important;
        padding: 5px !important;
    }
    </style>
    """, unsafe_allow_html=True)

# 3. Render Premium Logo Title
st.markdown(
    "<div style='overflow: visible; padding: 10px 0; margin-bottom: 5px;'>"
    "<span style='"
    "font-family: \"Inter\", system-ui, sans-serif; "
    "font-weight: 700; "
    "font-size: 2.8rem; "
    "display: inline-block; "
    "line-height: 1.6; "
    "background: linear-gradient(90deg, #0076ff, #00f2fe); "
    "-webkit-background-clip: text; "
    "-webkit-text-fill-color: transparent; "
    "color: #0076ff !important; "
    "text-shadow: 0px 0px 15px rgba(0, 118, 255, 0.2);"
    "'>Qwen AI Assistant</span>"
    "</div>",
    unsafe_allow_html=True
)
st.divider()

# 4. Sidebar Architecture - Styled Privately & Premium
with st.sidebar:
    # Updated: Rendered with premium grey style class
    st.markdown('<p class="sidebar-header-premium">📁 Context Ingestion</p>', unsafe_allow_html=True)
    st.markdown('<p class="sidebar-subheader-premium">Drop target file here</p>', unsafe_allow_html=True)

    uploaded_file = st.file_uploader(
        "Upload PDF or TXT",
        label_visibility="collapsed",
        type=['pdf', 'txt'],
        help="Upload small files for faster processing."
    )

    # Custom Warning Box with Yellow Flash
    st.markdown("""
        <div class="image-style-warning-box">
            <p class="image-style-warning-title">⚡ Engine constraint warning</p>
            <p class="image-style-warning-text">Free CPU slots run index engines slowly. Limit document size to <b>1-5 pages</b> for live processing.</p>
        </div>
        """, unsafe_allow_html=True)

    if uploaded_file:
        if not os.path.exists("./data"):
            os.makedirs("./data")
        for f in os.listdir("./data"):
            os.remove(os.path.join("./data", f))
        file_path = os.path.join("./data", uploaded_file.name)
        with open(file_path, "wb") as f:
            f.write(uploaded_file.getbuffer())
        st.success("File uploaded successfully")

    st.divider()

    # Premium Live Status Panel Box
    status_icon = "🟢" if "index" in st.session_state else "🔴"
    status_label = "System Active" if "index" in st.session_state else "Awaiting Document Upload"

    st.markdown(f"""
        <div class="status-badge-container">
            <p style='margin:0; font-size:0.82rem; color:#8b949e; font-weight:500;'>CORE ENGINE STATUS</p>
            <p style='margin:5px 0 0 0; font-size:0.95rem; color:#f0f6fc; font-weight:600;'>{status_icon} {status_label}</p>
        </div>
        """, unsafe_allow_html=True)

    if st.button("Build Knowledge Base", use_container_width=True):
        with st.spinner("Executing structural chunking..."):
            st.session_state.index = process_documents()
            st.success("🎯 Synapse network ready.")
            st.rerun()

# 5. Chat History Display
if "messages" not in st.session_state:
    st.session_state.messages = []

# Loop through history and render using Streamlit chat bubbles with correct avatars
for msg in st.session_state.messages:
    if msg["role"] == "user":
        with st.chat_message("user", avatar="👤"):
            st.write(msg['content'])
    else:
        with st.chat_message("assistant", avatar="👾"):
            st.write(msg['content'])

# 6. Core Process & Streaming Text Integration
if prompt := st.chat_input("Dispatch query into local workspace..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user", avatar="👤"):
        st.write(prompt)

    if "index" in st.session_state:
        with st.chat_message("assistant", avatar="👾"):
            with st.spinner("Searching knowledge base..."):
                qa_prompt_str = (
                    "Context information is below.\n"
                    "---------------------\n"
                    "{context_str}\n"
                    "---------------------\n"
                    "Answer ONLY using the provided context. If the answer is NOT in the context, "
                    "strictly say: 'I am sorry, but this information is not in the uploaded document.'\n"
                    "Query: {query_str}\n"
                    "Answer: "
                )
                qa_template = PromptTemplate(qa_prompt_str)
                query_engine = st.session_state.index.as_query_engine(streaming=True, text_qa_template=qa_template)
                response = query_engine.query(prompt)

            # Response Typewriter Streaming Effect
            placeholder = st.empty()
            full_response = ""
            for text in response.response_gen:
                full_response += text
                placeholder.markdown(full_response + "▌")
            placeholder.markdown(full_response)

            st.session_state.messages.append({"role": "assistant", "content": full_response})
            st.rerun()
    else:
        st.warning("⚠️ Access denied. Please upload a document and build the knowledge base in the sidebar first.")


Overwriting app.py


In [49]:
import os, time, subprocess, requests

# Cleanup
!pkill -9 streamlit
!pkill -9 cloudflared

# Download cloudflared if needed
if not os.path.exists("cloudflared"):
    !wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# Start Streamlit
print("Starting Streamlit...")
os.system("nohup python3 -m streamlit run app.py --server.port 8501 --server.address 127.0.0.1 --server.headless true > streamlit.log 2>&1 &")

# Ping to ensure it's up
for i in range(15):
    try:
        if requests.get("http://127.0.0.1:8501").status_code == 200:
            print("\n✅Streamlit is UP!")
            break
    except:
        print(".", end=""); time.sleep(5)

# Tunnel
print("\n🔗 Click the link below:")
!./cloudflared tunnel --url http://127.0.0.1:8501

Starting Streamlit...
.
✅Streamlit is UP!

🔗 Click the link below:
2026-06-20T17:22:34Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-20T17:22:34Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-20T17:22:37Z INF +--------------------------------------------------------------------------------------------+
2026-06-20T17:22:37Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-20T17: